# Verify splits — leakage audit & reference floor

Converted from `05_verify_splits.py`. Two independent things happen here:

1. **Leakage audit** (the point of the notebook). For every split folder: no
   text may appear in two splits, no parent thread may straddle a boundary,
   and for `sarc_subreddit` no subreddit may either. A failure here
   invalidates every number produced downstream, so this is a **hard gate**,
   not just a report.
2. **Reference floor** (a sanity check, **not** the project's baseline). A
   TF-IDF + logistic-regression fit, so we know the data is learnable and
   roughly where the floor sits. This is a smoke test that the splits load
   and behave — the real base models are assigned separately in Week 7.
3. **Cross-corpus transfer** — train once on SARC, score on every other
   corpus's test split. The cheapest version of the sub-question-4 number.

Set `AUDIT_ONLY = True` below to skip the floor/transfer fits (equivalent to
the original `--audit-only` flag) and just run the leakage checks.


## Setup

In [11]:
from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd() / ".."))  # adjust if this notebook lives elsewhere
from common.paths import DS_PRIMARY, DS_SUPP, SPLIT_RESULTS  # noqa: E402
from common import text_cleaning as tc                                # noqa: E402

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

In [12]:
SEED = 3244
SPLITS = ("train", "val", "test")

## Config

Equivalent to the script's `--audit-only` / `--max-train` CLI flags.

In [13]:
AUDIT_ONLY = False      # True = skip the reference floor and transfer fits
MAX_TRAIN = 200_000     # cap on training rows for the reference floor


## Helpers

In [14]:
def load_split(d: Path) -> dict[str, pd.DataFrame]:
    return {s: pd.read_parquet(d / f"{s}.parquet")
            for s in SPLITS if (d / f"{s}.parquet").exists()}

In [15]:
def text_col(df: pd.DataFrame) -> str:
    return "comment_clean" if "comment_clean" in df.columns else "text_clean"

## 1. Leakage audit

Checks, per split folder:
- no cleaned text appears in more than one of train/val/test
- no parent-comment thread straddles a split boundary
- (for `sarc_subreddit` only) no subreddit appears in more than one split

Author overlap is *reported, never failed*: SARC has 254k authors over 914k
rows, so disjoint authors aren't achievable without gutting the corpus — it's
recorded so the write-up can acknowledge it rather than silently ignore it.

In [16]:
def audit(name: str, d: Path) -> dict:
    parts = load_split(d)
    if not parts:
        return {"dataset": name, "status": "MISSING", "checks": []}

    col = text_col(next(iter(parts.values())))
    checks, failed = [], False

    def add(check: str, bad: int, detail: str = "") -> None:
        nonlocal failed
        ok = bad == 0
        failed = failed or not ok
        checks.append({"check": check, "overlap": int(bad),
                       "status": "PASS" if ok else "FAIL", "detail": detail})

    keys = {s: set(tc.dedup_key(p[col])) for s, p in parts.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        if a in keys and b in keys:
            add(f"text overlap {a}/{b}", len(keys[a] & keys[b]))

    if "parent_clean" in next(iter(parts.values())).columns:
        pk = {s: set(p["parent_clean"].fillna("").str.strip().str.lower())
              for s, p in parts.items()}
        for s in pk:                       # the empty parent is not a thread
            pk[s].discard("")
        for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
            if a in pk and b in pk:
                add(f"parent-thread overlap {a}/{b}", len(pk[a] & pk[b]))

    if name == "sarc_subreddit":
        sr = {s: set(p["subreddit"]) for s, p in parts.items()}
        for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
            add(f"subreddit overlap {a}/{b}", len(sr[a] & sr[b]))

    # Author overlap is reported, never failed: SARC has 254k authors over
    # 914k rows, so disjoint authors are not achievable without gutting the
    # corpus. It is recorded so the write-up can acknowledge it.
    author_note = ""
    if "author" in next(iter(parts.values())).columns:
        au = {s: set(p["author"]) for s, p in parts.items()}
        if "train" in au and "test" in au:
            shared = len(au["train"] & au["test"])
            author_note = (f"{shared:,} authors appear in both train and test "
                           f"({100*shared/max(1,len(au['test'])):.0f}% of test "
                           f"authors) — inherent to SARC, disclose in the report")

    stats = {s: {"rows": int(len(p)),
                 "sarcastic_pct": round(float(p["label"].mean()) * 100, 2)}
             for s, p in parts.items()}

    return {"dataset": name, "status": "FAIL" if failed else "PASS",
            "checks": checks, "splits": stats, "author_note": author_note}

## 2. Reference floor

TF-IDF (1-2 grams) + untuned logistic regression on each split's train/test.
Deliberately not tuned — the point is only to confirm the files load and are
learnable, and to give the real Week-7 models something to beat.

In [17]:
def floor(name: str, d: Path, max_train: int) -> dict | None:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    from sklearn.dummy import DummyClassifier

    parts = load_split(d)
    if "train" not in parts or "test" not in parts:
        return None
    col = text_col(parts["train"])

    tr = parts["train"]
    if len(tr) > max_train:
        tr = tr.sample(max_train, random_state=SEED)
    te = parts["test"]

    t0 = time.time()
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000,
                          sublinear_tf=True, strip_accents="unicode")
    Xtr = vec.fit_transform(tr[col].fillna(""))
    Xte = vec.transform(te[col].fillna(""))

    dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, tr["label"])
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(Xtr, tr["label"])
    pred = clf.predict(Xte)

    res = {
        "dataset": name,
        "train_rows_used": int(len(tr)),
        "test_rows": int(len(te)),
        "majority_class_accuracy": round(float(accuracy_score(
            te["label"], dummy.predict(Xte))), 4),
        "tfidf_logreg": {
            "accuracy": round(float(accuracy_score(te["label"], pred)), 4),
            "precision": round(float(precision_score(te["label"], pred, zero_division=0)), 4),
            "recall": round(float(recall_score(te["label"], pred, zero_division=0)), 4),
            "f1": round(float(f1_score(te["label"], pred, zero_division=0)), 4),
        },
        "n_features": int(Xtr.shape[1]),
        "fit_seconds": round(time.time() - t0, 1),
    }
    m = res["tfidf_logreg"]
    print(f"  [{name:<18}] acc={m['accuracy']:.4f}  P={m['precision']:.4f}  "
          f"R={m['recall']:.4f}  F1={m['f1']:.4f}   "
          f"(majority {res['majority_class_accuracy']:.4f}, {res['fit_seconds']}s)")
    return res

## 3. Cross-corpus transfer floor

Train once on `sarc_random`'s train split, then score that same model on
every other corpus's test split (including `sarc_random`'s own test split,
labelled in-domain) — the cheapest version of "does anything transfer off
Reddit?".

In [18]:
def transfer(max_train: int) -> list[dict]:
    """
    Train once on SARC, score on every other corpus's test split.

    This is the sub-question-4 number in its cheapest form. Having it now means
    the Week-7 models have something to beat, and it tells us in advance
    whether cross-platform transfer is a real experiment or a formality.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

    src = DS_PRIMARY / "sarc_random"
    if not (src / "train.parquet").exists():
        return []
    tr = pd.read_parquet(src / "train.parquet")
    if len(tr) > max_train:
        tr = tr.sample(max_train, random_state=SEED)

    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000,
                          sublinear_tf=True, strip_accents="unicode")
    Xtr = vec.fit_transform(tr["comment_clean"].fillna(""))
    clf = LogisticRegression(max_iter=1000, C=1.0).fit(Xtr, tr["label"])

    out = []
    targets = [("sarc_random (in-domain)", DS_PRIMARY / "sarc_random" / "test.parquet")]
    targets += [(p.name, p / "test.parquet") for p in sorted(DS_SUPP.iterdir())
                if p.is_dir() and (p / "test.parquet").exists()]

    for name, path in targets:
        te = pd.read_parquet(path)
        col = text_col(te)
        pred = clf.predict(vec.transform(te[col].fillna("")))
        r = {
            "target": name,
            "test_rows": int(len(te)),
            "accuracy": round(float(accuracy_score(te["label"], pred)), 4),
            "precision": round(float(precision_score(te["label"], pred, zero_division=0)), 4),
            "recall": round(float(recall_score(te["label"], pred, zero_division=0)), 4),
            "f1": round(float(f1_score(te["label"], pred, zero_division=0)), 4),
            "majority_baseline": round(float(max(te["label"].mean(),
                                                 1 - te["label"].mean())), 4),
        }
        out.append(r)
        print(f"  SARC -> {name:<26} acc={r['accuracy']:.4f}  F1={r['f1']:.4f}   "
              f"(majority {r['majority_baseline']:.4f})")
    return out

## Run the leakage audit

Gathers every split folder under `DS_PRIMARY` and `DS_SUPP`, runs `audit` on each, and prints a PASS/FAIL summary.

In [19]:
targets = [(p.name, p) for p in sorted(DS_PRIMARY.iterdir()) if p.is_dir()]
targets += [(p.name, p) for p in sorted(DS_SUPP.iterdir()) if p.is_dir()]

print("=== LEAKAGE AUDIT ===")
audits, n_fail = [], 0
for name, d in targets:
    a = audit(name, d)
    audits.append(a)
    bad = [c for c in a["checks"] if c["status"] == "FAIL"]
    n_fail += len(bad)
    print(f"  [{a['status']}] {name:<18} {len(a['checks'])} checks"
          + (f"  <-- {len(bad)} FAILED" if bad else ""))
    for c in bad:
        print(f"        FAIL {c['check']}: {c['overlap']:,} overlapping")
    if a.get("author_note"):
        print(f"        note: {a['author_note']}")


=== LEAKAGE AUDIT ===
  [PASS] sarc_random        6 checks
        note: 78,836 authors appear in both train and test (84% of test authors) — inherent to SARC, disclose in the report
  [PASS] sarc_subreddit     9 checks
        note: 55,446 authors appear in both train and test (72% of test authors) — inherent to SARC, disclose in the report
  [PASS] sarc_temporal      6 checks
        note: 41,379 authors appear in both train and test (36% of test authors) — inherent to SARC, disclose in the report
  [PASS] figlang_reddit     3 checks
  [PASS] figlang_twitter    3 checks
  [PASS] news_headlines     3 checks
  [PASS] tweeteval_irony    3 checks


## Run the reference floor

Skipped if `AUDIT_ONLY = True`.

In [20]:
floors = []
if not AUDIT_ONLY:
    print("\n=== REFERENCE FLOOR (TF-IDF + logistic regression, untuned) ===")
    print("  NOT the project baseline — a smoke test that the splits load "
          "and are learnable.")
    for name, d in targets:
        try:
            r = floor(name, d, MAX_TRAIN)
            if r:
                floors.append(r)
        except Exception as exc:
            print(f"  [{name}] floor failed: {exc}")



=== REFERENCE FLOOR (TF-IDF + logistic regression, untuned) ===
  NOT the project baseline — a smoke test that the splits load and are learnable.
  [sarc_random       ] acc=0.7058  P=0.7231  R=0.6868  F1=0.7045   (majority 0.5107, 23.3s)
  [sarc_subreddit    ] acc=0.7016  P=0.7138  R=0.6719  F1=0.6922   (majority 0.4993, 10.9s)
  [sarc_temporal     ] acc=0.6941  P=0.6563  R=0.6968  F1=0.6759   (majority 0.4579, 15.8s)
  [figlang_reddit    ] acc=0.6054  P=0.6154  R=0.5498  F1=0.5808   (majority 0.5028, 0.2s)
  [figlang_twitter   ] acc=0.6449  P=0.6024  R=0.7436  F1=0.6656   (majority 0.4752, 0.3s)
  [news_headlines    ] acc=0.8344  P=0.8171  R=0.8393  F1=0.8281   (majority 0.5248, 0.8s)
  [tweeteval_irony   ] acc=0.6372  P=0.5354  R=0.6581  F1=0.5904   (majority 0.3974, 0.2s)


## Run the cross-corpus transfer check

Skipped if `AUDIT_ONLY = True`.

In [21]:
transfers = []
if not AUDIT_ONLY:
    print("\n=== CROSS-CORPUS TRANSFER (train on SARC, test elsewhere) ===")
    try:
        transfers = transfer(MAX_TRAIN)
    except Exception as exc:
        print(f"  transfer check failed: {exc}")



=== CROSS-CORPUS TRANSFER (train on SARC, test elsewhere) ===
  SARC -> sarc_random (in-domain)    acc=0.7058  F1=0.7045   (majority 0.5107)
  SARC -> figlang_reddit             acc=0.6686  F1=0.6608   (majority 0.5028)
  SARC -> figlang_twitter            acc=0.6140  F1=0.5384   (majority 0.5248)
  SARC -> news_headlines             acc=0.4599  F1=0.3131   (majority 0.5248)
  SARC -> tweeteval_irony            acc=0.6154  F1=0.4403   (majority 0.6026)


## Save results

In [22]:
SPLIT_RESULTS.mkdir(parents=True, exist_ok=True)
payload = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "audit": audits,
    "reference_floor": floors,
    "cross_corpus_transfer": transfers,
    "audit_failures": n_fail,
}
p = SPLIT_RESULTS / "split_verification.json"
p.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n[save] {p}")
print(f"{'ALL CHECKS PASSED' if n_fail == 0 else str(n_fail) + ' CHECK(S) FAILED'}")



[save] C:\Users\USER\OneDrive\Documents\CS3244\cs3244_group1\results\splits\split_verification.json
ALL CHECKS PASSED
